# 01 : Datos y exploración

Se descargan o reutilizan los siete CSV de Store Sales, se comprueban sus esquemas y se exploran las diferencias entre fechas, tiendas y familias. La variable objetivo es venta observada, no demanda perdida por falta de inventario.

## Ejecución de la entrega

Ejecuta todas las celdas en orden en una sesión limpia. El modo predeterminado
`full` utiliza datos reales; el notebook 04 requiere GPU. Para comprobar solo
esquema y ejecución, define `os.environ["RETAIL_FORECAST_MODE"] = "smoke"` antes
de la preparación. Los resultados sintéticos no miden precisión.

Mantén los siete CSV en `data/raw` del espacio de trabajo o usa el ZIP de Kaggle.
No escribas credenciales en celdas ni salidas. Conserva el notebook ejecutado y
el directorio de resultados antes de desconectar Colab.

## Cómo leer este notebook

Las cifras de interpretación se refieren a los datos reales de la competencia. En modo `smoke`, las figuras son sintéticas y esas cifras no se aplican. La exploración describe asociaciones; las decisiones de predicción se evalúan después en ventanas temporales.

In [ ]:
# Pinned source setup; Colab preserves its compatible resident libraries.
import importlib.util
import os
import re
import runpy
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path

# The published academic URL can be supplied through the environment.
REPO_URL = os.environ.get("RETAIL_FORECAST_REPO_URL", "").strip() or globals().get("REPO_URL", "")
if not REPO_URL:
    if (Path.cwd() / "pyproject.toml").is_file():
        REPO_URL = str(Path.cwd())
    else:
        REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
if not REPO_URL:
    raise ValueError("Indica la URL del repositorio académico publicado.")
CODE_REVISION = "af4d384e8fa47b6034f76aec16e3b598c4fe0d98"
MODELS = False
if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.10–3.13 before installing or acquiring data.")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise RuntimeError("CODE_REVISION must be a reviewed full commit SHA.")
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
REPO_ROOT = Path(
    os.environ.get(
        "RETAIL_FORECAST_CHECKOUT",
        str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / f"retail-academic-{CODE_REVISION}"),
    )
).absolute()
if REPO_ROOT.is_symlink() or any(p.is_symlink() for p in REPO_ROOT.parents):
    raise RuntimeError("Checkout path must not contain symlinks; use a new directory.")


def checkout_git(*arguments):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], check=True)
    # A squash merge can leave the reviewed SHA reachable only by its PR ref.
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--no-tags", "origin", CODE_REVISION],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION],
        check=True,
    )
# Validate before executing even the shared helper from this checkout.
try:
    valid = (
        checkout_git("rev-parse", "--show-toplevel") == str(REPO_ROOT)
        and checkout_git("remote", "get-url", "origin") == REPO_URL
        and checkout_git("rev-parse", "HEAD") == CODE_REVISION
        and not checkout_git("status", "--porcelain", "--untracked-files=all")
    )
except subprocess.CalledProcessError:
    valid = False
if not valid:
    raise RuntimeError(
        "Dirty, incomplete or mismatched checkout. Preserve it; use a new directory."
    )
helper = runpy.run_path(str(REPO_ROOT / "src/retail_forecast/notebook_runtime.py"))
runtime_evidence = helper["bootstrap"](
    REPO_ROOT, CODE_REVISION, REPO_URL, models=MODELS, colab=IN_COLAB
)
print(
    "Notebook revision:",
    os.getenv("RETAIL_FORECAST_NOTEBOOK_REVISION", "unrecorded; retain immutable notebook URL"),
)
WORKSPACE_ROOT = globals().get(
    "WORKSPACE_ROOT",
    Path("/content") if IN_COLAB else (
        Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else REPO_ROOT
    ),
)
os.chdir(REPO_ROOT)

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from retail_forecast import __version__
from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.artifacts import (
    capture_inputs,
    finish_run,
    read_completed_run,
    run_result,
    write_run_artifact,
)
from retail_forecast.config import load_config
from retail_forecast.data import required_files_exist, validate_dataset, write_synthetic_dataset
from retail_forecast.eda import (
    build_eda_summary,
    build_eda_tables,
    build_manifest,
    export_eda_artifacts,
    load_eda_frames,
    save_figure,
    validate_kaggle_hashes,
)
from retail_forecast.logging_utils import configure_logging
from retail_forecast.notebook_runs import prepare_notebook_run
from retail_forecast.reproducibility import set_global_seed
from retail_forecast.versions import capture_versions

MODE = os.getenv("RETAIL_FORECAST_MODE", "full").strip().lower()
if MODE not in {"full", "smoke"}:
    raise ValueError("RETAIL_FORECAST_MODE must be 'full' or 'smoke'.")
config = load_config(f"configs/{MODE}.yaml")
config = replace(
    config,
    paths=replace(
        config.paths,
        raw_data_dir=WORKSPACE_ROOT / config.paths.raw_data_dir.relative_to(REPO_ROOT),
        artifacts_dir=WORKSPACE_ROOT / config.paths.artifacts_dir.relative_to(REPO_ROOT),
    ),
)
run_context, config = prepare_notebook_run(config, kind="audit")
print("Run ID:", run_context.run_id, "| root:", run_context.root)
configure_logging(config.runtime.log_level)
set_global_seed(config.runtime.seed)
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.titleweight": "bold"})
print(f"mode={MODE} | package={__version__} | horizon={config.forecast.horizon}")
display(capture_versions())

## Disponibilidad de los archivos

Coloca el ZIP en `/content/store-sales-time-series-forecasting.zip` o los CSV en `/content/data/raw`. Si no están disponibles, la celda ofrece descarga autenticada con entrada oculta. Los datos de Kaggle requieren acceso personal a la competencia.

In [ ]:
if config.data.source == "synthetic":
    if not required_files_exist(config.paths.raw_data_dir):
        write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir,
        (WORKSPACE_ROOT / "data", WORKSPACE_ROOT),
        token_reader=getpass,
    )

capture_inputs(run_context, input_dir=config.paths.raw_data_dir)
hashes = validate_kaggle_hashes(config.paths.raw_data_dir) if MODE == "full" else {}
dataset_contract = validate_dataset(config)
print("Acquisition and integrity checks: PASS")
display(dataset_contract.to_dict())

In [ ]:
frames = load_eda_frames(config.paths.raw_data_dir)
manifest = build_manifest(config.paths.raw_data_dir, frames)
tables = build_eda_tables(frames)
eda_summary = build_eda_summary(frames, tables, config.data.source)

METRICS_DIR = run_context.path("metrics")
FIGURES_DIR = run_context.path("figures/eda")
summary_path, manifest_path = export_eda_artifacts(
    eda_summary, manifest, METRICS_DIR, run=run_context
)
manifest_view = pd.DataFrame(
    [
        {
            "file": filename,
            "bytes": values["bytes"],
            "rows": values["rows"],
            "columns": len(values["columns"]),
            "missing": sum(values["missing"].values()),
            "date_min": values.get("date_min"),
            "date_max": values.get("date_max"),
        }
        for filename, values in manifest.items()
    ]
)
display(manifest_view)
print(f"Detailed dtypes and missingness exported to {manifest_path}")
write_run_artifact(
    run_context,
    "metrics/source_overview.csv",
    lambda path: manifest_view.to_csv(path, index=False),
    kind="audit_table",
)

## Evolución de las ventas agregadas

In [ ]:
daily = tables["daily_sales"]
fig, ax = plt.subplots()
ax.plot(daily["date"], daily["sales"], alpha=0.35, linewidth=0.8, label="Daily total")
ax.plot(daily["date"], daily["rolling_28d"], linewidth=2, label="28-day median")
ax.set(title="Daily sales and local level", xlabel="Date", ylabel="Units")
ax.legend()
save_figure(fig, FIGURES_DIR, "01_daily_sales.png", run=run_context)
plt.show()

level_ratio = eda_summary["observations"]["last_to_first_90d_ratio"]
print(f"Last 90 days / first 90 days mean-sales ratio: {level_ratio:.3f}x")

### Cambio de nivel

La media diaria de los últimos 90 días es 2,451 veces la de los primeros 90. La curva y la mediana móvil permiten ver el cambio, pero no prueban formalmente estacionariedad. La evaluación se hace por tiempo y las transformaciones se ajustan dentro del entrenamiento de cada corte.

## Ceros y valores grandes

In [ ]:
sales = frames["train.csv"]["sales"]
fig, ax = plt.subplots()
ax.hist(np.log1p(sales), bins=60, color="#4472C4", alpha=0.85)
ax.set(title="Distribution of log1p(sales)", xlabel="log1p(sales)", ylabel="Rows")
save_figure(fig, FIGURES_DIR, "02_sales_distribution.png", run=run_context)
plt.show()

target_stats = eda_summary["observations"]
display(
    pd.Series(
        {
            "Zero-sales rows": f"{target_stats['zero_sales_rate']:.2%}",
            "Rows above the IQR rule": f"{target_stats['iqr_outlier_candidates']:,}",
            "Fractional-sales rows": f"{target_stats['fractional_sales_rows']:,}",
            "Median sales": f"{target_stats['sales_quantiles']['0.5']:,.2f}",
            "99th percentile": f"{target_stats['sales_quantiles']['0.99']:,.2f}",
            "Maximum": f"{target_stats['sales_quantiles']['1.0']:,.2f}",
        },
        name="Measured value",
    ).to_frame()
)

### Distribución del objetivo

El 31,30 % de las observaciones tiene venta cero. Los valores grandes no se eliminan por superar el criterio IQR: pueden corresponder a ventas válidas. Se utiliza `log1p` y se conservan los ceros; una fecha ausente se trata aparte, con una máscara de observación.

## Diferencias de volumen entre tiendas

In [ ]:
store_sales = tables["store_sales"].sort_values("sales")
figure_height = max(5.0, 0.24 * len(store_sales))
fig, ax = plt.subplots(figsize=(11, figure_height))
ax.barh(store_sales["store_nbr"].astype(str), store_sales["sales"], color="#70AD47")
ax.set(title="Total sales by store", xlabel="Units", ylabel="Store")
ax.tick_params(axis="y", labelsize=8)
ax.margins(y=0.01)
save_figure(fig, FIGURES_DIR, "03_store_sales.png", run=run_context)
plt.show()

smallest_store = store_sales.iloc[0]
largest_store = store_sales.iloc[-1]
store_ratio = largest_store["sales"] / max(smallest_store["sales"], 1e-12)
display(
    pd.Series(
        {
            "Largest cumulative-sales store": int(largest_store["store_nbr"]),
            "Smallest cumulative-sales store": int(smallest_store["store_nbr"]),
            "Largest / smallest total": f"{store_ratio:.2f}x",
        },
        name="Measured value",
    ).to_frame()
)

### Escalas de venta

La distribución de ventas acumuladas entre tiendas es desigual. Esto motiva compartir un modelo global con identificación de tienda y examinar sus errores por local; el volumen acumulado, por sí solo, no indica qué tienda será más difícil de pronosticar.

## Volumen y ceros por familia

In [ ]:
family_sales = tables["family_sales"].head(15).sort_values("sales")
fig, ax = plt.subplots()
ax.barh(family_sales["family"], family_sales["sales"], color="#ED7D31")
ax.set(title="Top families by total sales", xlabel="Units", ylabel="Family")
save_figure(fig, FIGURES_DIR, "04_family_sales.png", run=run_context)
plt.show()

all_family = tables["family_sales"]
top_family = all_family.iloc[0]
highest_zero = all_family.sort_values("zero_rate", ascending=False).iloc[0]
display(
    pd.Series(
        {
            "Largest cumulative-sales family": top_family["family"],
            "Highest-zero family": highest_zero["family"],
            "Highest family zero rate": f"{highest_zero['zero_rate']:.2%}",
        },
        name="Measured value",
    ).to_frame()
)

### Representación de familias

BOOKS tiene 96,96 % de ventas cero en los datos reales. Una métrica agrupada puede esconder comportamientos distintos por familia. Se mantendrán los errores por familia en las comparaciones, sin concluir que más ceros impliquen automáticamente mayor RMSLE.

## Promociones y ventas observadas

In [ ]:
promotion = tables["promotion"].copy()
labels = promotion["promoted"].map({False: "No promotion", True: "Promoted"})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(labels, promotion["mean_sales"], color=["#A5A5A5", "#FFC000"])
axes[0].set(title="Mean sales", ylabel="Units")
axes[1].bar(labels, promotion["zero_rate"], color=["#A5A5A5", "#FFC000"])
axes[1].set(title="Zero-sales rate", ylabel="Share")
fig.suptitle("Promotion association")
save_figure(fig, FIGURES_DIR, "05_promotions.png", run=run_context)
plt.show()

promotion_view = promotion.assign(group=labels).set_index("group")
promotion_ratio = promotion_view.loc["Promoted", "mean_sales"] / max(
    promotion_view.loc["No promotion", "mean_sales"], 1e-12
)
promotion_metrics = promotion_view[["rows", "mean_sales", "median_sales", "zero_rate"]].copy()
promotion_metrics["rows"] = promotion_metrics["rows"].map(lambda value: f"{value:,.0f}")
promotion_metrics["mean_sales"] = promotion_metrics["mean_sales"].map(lambda value: f"{value:,.2f}")
promotion_metrics["median_sales"] = promotion_metrics["median_sales"].map(
    lambda value: f"{value:,.2f}"
)
promotion_metrics["zero_rate"] = promotion_metrics["zero_rate"].map(lambda value: f"{value:.2%}")
display(promotion_metrics)
print(f"Promoted / non-promoted mean-sales ratio: {promotion_ratio:.2f}x")

### Asociación con promociones

Las filas con promoción tienen otro nivel de ventas y otra proporción de ceros. La comparación mezcla tiendas, familias y fechas: no estima el efecto causal de promover un producto. En la evaluación se usan promociones históricas como aproximación al plan que habría sido conocido al emitir el pronóstico.

## Transacciones y ventas agregadas

In [ ]:
sales_transactions = tables["sales_transactions"]
fig, ax = plt.subplots()
ax.scatter(sales_transactions["transactions"], sales_transactions["sales"], s=12, alpha=0.35)
ax.set(title="Daily transactions and sales", xlabel="Transactions", ylabel="Units sold")
save_figure(fig, FIGURES_DIR, "06_transactions.png", run=run_context)
plt.show()

transaction_correlation = sales_transactions[["transactions", "sales"]].corr().iloc[0, 1]
transaction_coverage = eda_summary["observations"]["transaction_store_date_coverage"]
print(f"Aggregate Pearson correlation: {transaction_correlation:.3f}")
print(f"Transaction store-date coverage: {transaction_coverage:.2%}")

### Información histórica de transacciones

Las dos magnitudes se relacionan, aunque el diagrama no forma una única nube estable. Las transacciones de los 16 días futuros no se conocen al origen y no se incorporan. Solo se usan sus valores históricos y resúmenes anteriores al corte.

## Petróleo y ventas: exploración histórica

In [ ]:
oil_sales = tables["oil_sales"].dropna(subset=["dcoilwtico"])
fig, ax = plt.subplots()
ax.scatter(oil_sales["dcoilwtico"], oil_sales["sales"], s=12, alpha=0.35, color="#5B9BD5")
ax.set(title="Oil price and daily sales", xlabel="WTI price", ylabel="Units sold")
save_figure(fig, FIGURES_DIR, "07_oil.png", run=run_context)
plt.show()

oil_correlation = oil_sales[["dcoilwtico", "sales"]].corr().iloc[0, 1]
oil_missing = eda_summary["observations"]["oil_missing_values"]
print(f"Contemporaneous aggregate Pearson correlation: {oil_correlation:.3f}")
print(f"Missing oil values: {oil_missing:,}")

### Alcance de esta asociación

La asociación histórica con petróleo mezcla cambios de precio y cambios de nivel de ventas. No permite atribuirle un efecto causal. La política principal `asof_v1` excluye el petróleo por completo. La política histórica `competition_v1` sí usaba valores suministrados para las fechas pronosticadas; sus resultados se conservan separados.

## Festivos, eventos y uniones por fecha

In [ ]:
holiday_types = tables["holiday_types"].sort_values("size")
fig, ax = plt.subplots()
ax.barh(holiday_types["type"], holiday_types["size"], color="#8064A2")
ax.set(title="Holiday and event records by type", xlabel="Rows", ylabel="Type")
save_figure(fig, FIGURES_DIR, "08_holidays.png", run=run_context)
plt.show()

duplicate_holiday_rows = eda_summary["observations"]["holiday_duplicate_date_rows"]
print(f"Holiday rows on duplicated dates: {duplicate_holiday_rows:,}")

### Calendario por tienda

Una fecha puede contener varias filas y ámbitos geográficos. El calendario se resuelve por ciudad y estado antes de unirlo a la tienda. En `asof_v1` se excluyen todas las filas de tipo `Event`; los demás tipos se tratan como programados. No se dispone de versiones históricas del calendario para comprobar perfectamente esa disponibilidad.

## Patrones de calendario en entrenamiento

In [ ]:
weekday = tables["weekday"]
month = tables["month"]
annual_cycle = tables["annual_cycle"]
weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar([weekday_names[int(value)] for value in weekday["weekday"]], weekday["sales"])
axes[0].set(title="Weekly", ylabel="Mean daily units")
axes[1].bar(month["month"].astype(str), month["sales"], color="#70AD47")
axes[1].set(title="Monthly", xlabel="Month")
axes[2].plot(annual_cycle["dayofyear"], annual_cycle["sales"], color="#C00000")
axes[2].set(title="Annual cycle", xlabel="Day of year")
fig.suptitle("Seasonal patterns measured from training data only")
save_figure(fig, FIGURES_DIR, "09_seasonality.png", run=run_context)
plt.show()

best_weekday = weekday.loc[weekday["sales"].idxmax(), "weekday"]
best_month = month.loc[month["sales"].idxmax(), "month"]
best_annual_day = annual_cycle.loc[annual_cycle["sales"].idxmax(), "dayofyear"]
print(f"Highest mean-sales weekday: {weekday_names[int(best_weekday)]}")
print(f"Highest mean-sales month: {int(best_month)}")
print(f"Highest mean-sales day of year: {int(best_annual_day)}")

### Lectura de la estacionalidad

Los perfiles por día de semana y mes sugieren estructura de calendario, pero agregan años y tiendas con escalas diferentes. Se incorporan esas variables y se compara con referencias semanales. Las diferencias del gráfico no bastan para afirmar que una fecha concreta cause una subida de ventas.

## Comprobaciones finales

In [ ]:
final_manifest = build_manifest(config.paths.raw_data_dir, frames)
raw_unchanged = all(manifest[name]["sha256"] == final_manifest[name]["sha256"] for name in manifest)
figure_files = sorted(FIGURES_DIR.glob("*.png"))
checks = {
    "16-day horizon": dataset_contract.horizon == 16,
    "seven schemas loaded": len(frames) == 7,
    "Kaggle hashes verified or synthetic mode": MODE == "smoke" or len(hashes) == 7,
    "raw files unchanged during EDA": raw_unchanged,
    "EDA summary exported": summary_path.is_file(),
    "dataset manifest exported": manifest_path.is_file(),
    "figure limit respected": 0 < len(figure_files) <= 10,
    "future transactions absent": frames["transactions.csv"]["date"].max()
    <= frames["train.csv"]["date"].max(),
}
check_table = pd.DataFrame(
    [{"check": name, "status": "PASS" if passed else "FAIL"} for name, passed in checks.items()]
)
display(check_table)
if not all(checks.values()):
    raise AssertionError(
        f"EDA checks failed: {[name for name, passed in checks.items() if not passed]}"
    )
print(f"PASS: {len(checks)} checks; {len(figure_files)} figures; raw inputs unchanged.")
display(
    pd.Series(
        {
            "Mode": MODE,
            "Training rows": f"{dataset_contract.train_rows:,}",
            "Test rows": f"{dataset_contract.test_rows:,}",
            "Store-family series": f"{dataset_contract.series:,}",
            "Forecast horizon": f"{dataset_contract.horizon} days",
            "Missing training dates": len(eda_summary["observations"]["missing_calendar_dates"]),
        },
        name="Run result",
    ).to_frame()
)
print(f"Artifacts ready: {summary_path} and {FIGURES_DIR}")
finish_run(run_context)
verified_run = read_completed_run(
    run_context.root,
    expected_mode=MODE,
    expected_source=config.data.source,
)
print("Verified completed run:", run_result(run_context))

### Decisiones para los siguientes notebooks

Se conserva la estructura tienda–familia–fecha, se separan ceros y ausencias y se evalúan 16 días por origen. El siguiente notebook muestra qué información se admite al corte y cómo se ajustan las transformaciones sin usar el periodo de evaluación. Los resultados de los modelos se discuten en 03–05.